# IBKR API notebook

#### Connection

In [3]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from ib_async import *
import pandas as pd
util.startLoop()
global historical_data_interval, duration

ib = IB()
ib.connect('127.0.0.1', 7497, clientId=14)
if ib.isConnected():
    print("✅ Connected to IBKR API")
else:
    print("❌Failed to connect to IBKR API")
util.logToConsole(logging.INFO)

2025-06-04 09:43:43,672 - INFO - Connecting to 127.0.0.1:7497 with clientId 14...
2025-06-04 09:43:43,673 - INFO - Connected
2025-06-04 09:43:43,676 - INFO - Logged on to server version 178
2025-06-04 09:43:43,717 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfuture
2025-06-04 09:43:43,718 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarm
2025-06-04 09:43:43,718 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:cashfarm
2025-06-04 09:43:43,719 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:eufarmnj
2025-06-04 09:43:43,719 - INFO - Warning 2104, reqId -1: La connexion de donn\u00e9es de march\u00e9 est OK:usfarm
2025-06-04 09:43:43,719 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:euhmds
2025-06-04 09:43:43,719 - INFO - Warning 2106, reqId -1: La connexion de donn\u00e9es HMDS est OK:ushmds
2025-06-04 09:43:43

✅ Connected to IBKR API


## Request Historical data

#### Choose your contract

In [4]:
#contract = CFD('IBUST100', 'SMART', 'USD')
#contract = Forex(pair="EURUSD", exchange='IDEALPRO')
contract = Index('NDX', 'NASDAQ', 'USD')
contract_details = ib.reqContractDetails(contract)
# Extract and display the desired fields from contract_details
filtered_details = [
    {
        "secType": detail.contract.secType,
        "conId": detail.contract.conId,
        "symbol": detail.contract.symbol,
        "exchange": detail.contract.exchange,
        "longName": detail.longName,
        "timezoneId": detail.timeZoneId,
        "tradingHours": "\n".join(
            [f"  {segment}" for segment in detail.tradingHours.split(";")]
        ),
        "liquidHours": "\n".join(
            [f"  {segment}" for segment in detail.liquidHours.split(";")]
        ),
        "minSize": detail.minSize,
    }
    for detail in contract_details
]

# Print the filtered details in a clear format
for idx, detail in enumerate(filtered_details, start=1):
    print(f"Contract Detail {idx}:")
    for key, value in detail.items():
        print(f"  {key}: {value}")
    print()

Contract Detail 1:
  secType: IND
  conId: 416843
  symbol: NDX
  exchange: NASDAQ
  longName: NASDAQ 100 Stock Index
  timezoneId: US/Eastern
  tradingHours:   20250604:0930-20250604:1600
  20250605:0930-20250605:1600
  20250606:0930-20250606:1600
  20250607:CLOSED
  20250608:CLOSED
  20250609:0930-20250609:1600
  liquidHours:   20250604:0930-20250604:1600
  20250605:0930-20250605:1600
  20250606:0930-20250606:1600
  20250607:CLOSED
  20250608:CLOSED
  20250609:0930-20250609:1600
  minSize: 1.0



#### Check first data timestamp available

In [ ]:
timestamp = ib.reqHeadTimeStamp(contract, whatToShow='TRADES', useRTH=False)
formatted_time = timestamp.strftime('%B %d, %Y, %H:%M')
logging.info(f"First date of data available: {formatted_time}")

### Request historical data function

**End date choice**

In [ ]:
save_path = "./database/AAPL_10secs_20220131_to_20250403.parquet"
# Load the parquet file into a DataFrame
retrieved_df = pd.read_parquet(save_path)
# Retrieve the first date from your existing DataFrame
first_date = retrieved_df.iloc[0]['date']  # Assuming 'date' is the column name
end_date = first_date.strftime('%Y%m%d %H:%M:%S')  # Format as 'yyyyMMdd HH:mm:ss'
logging.info(f"First date in the DataFrame: {first_date}")
logging.info(f"End date: {end_date}")

In [ ]:
# yesterday's date
end_date = (pd.Timestamp.now(tz='UTC') - pd.DateOffset(days=1)).strftime('%Y%m%d %H:%M:%S')

In [5]:
#today's date
end_date = pd.Timestamp.now(tz='UTC').strftime('%Y%m%d-%H:%M:%S')

Args
- **contract**: Contract of interest.  
- **endDateTime**:  
    - Can be set to `''` to indicate the current time.  
    - Can be given as a `datetime.date` or `datetime.datetime`.  
    - Can be given as a string in `'yyyyMMdd HH:mm:ss'` format.  
    - If no timezone is given, the TWS login timezone is used.  
- **durationStr**: Time span of all the bars. Examples:  
    - `'60 S'`, `'30 D'`, `'13 W'`, `'6 M'`, `'10 Y'`.  
- **barSizeSetting**: Time period of one bar. Must be one of:  
    - `'1 secs'`, `'5 secs'`, `'10 secs'`, `'15 secs'`, `'30 secs'`,  
    - `'1 min'`, `'2 mins'`, `'3 mins'`, `'5 mins'`, `'10 mins'`, `'15 mins'`,  
    - `'20 mins'`, `'30 mins'`,  
    - `'1 hour'`, `'2 hours'`, `'3 hours'`, `'4 hours'`, `'8 hours'`,  
    - `'1 day'`, `'1 week'`, `'1 month'`.  
- **whatToShow**: Specifies the source for constructing bars. Must be one of:  
    - `'TRADES'`, `'MIDPOINT'`, `'BID'`, `'ASK'`, `'BID_ASK'`,  
    - `'ADJUSTED_LAST'`, `'HISTORICAL_VOLATILITY'`, `'OPTION_IMPLIED_VOLATILITY'`,  
    - `'REBATE_RATE'`, `'FEE_RATE'`, `'YIELD_BID'`, `'YIELD_ASK'`, `'YIELD_BID_ASK'`, `'YIELD_LAST'`.  
    - For `'SCHEDULE'`, use `:meth:.reqHistoricalSchedule`.  
- **useRTH**:  
    - If `True`, only show data from within Regular Trading Hours.  
    - If `False`, show all data.  
- **formatDate**:  
    - For an intraday request, setting to `2` will cause the returned date fields to be timezone-aware `datetime.datetime` with UTC timezone, instead of local timezone as used by TWS.  
- **keepUpToDate**:  
    - If `True`, a realtime subscription is started to keep the bars updated.  
    - `endDateTime` must be set empty (`''`) then.  
- **chartOptions**: Unknown.  
- **timeout**:  
    - Timeout in seconds after which to cancel the request and return an empty bar series. 
    - If the data request is huge, this parameter could spoil the request  
    - Set to `0` to wait indefinitely.  

In [6]:
historical_data_interval = '10 secs' 
request_duration = '30 D'  # Duration in days (use D, not "day")
price_source = 'TRADES'  # 'BID', 'ASK', or 'TRADES'
bars = ib.reqHistoricalData(
        contract,
        endDateTime=end_date,
        durationStr=str(request_duration),
        barSizeSetting=str(historical_data_interval),
        whatToShow=price_source,
        useRTH=False, 
        formatDate=2,
        timeout = 0)

Convert the list of bars to a data frame and print the first and last rows:

In [7]:
bars[0]
new_df = util.df(bars)

display(new_df.head())
display(new_df.tail())
# Remove the 'volume', 'average', and 'barCount' columns from the DataFrame
new_df = new_df.drop(columns=['volume', 'average', 'barCount'])

# Display the updated DataFrame
new_df.head()

,date,open,high,low,close,volume,average,barCount
0,2025-04-22 13:30:10+00:00,18033.90,18033.90,18019.76,18019.76,0.0,0.0,5
1,2025-04-22 13:30:20+00:00,18018.86,18018.86,18012.71,18012.74,0.0,0.0,10
2,2025-04-22 13:30:30+00:00,18013.64,18013.64,18002.60,18003.74,0.0,0.0,10
3,2025-04-22 13:30:40+00:00,18004.84,18007.12,18004.42,18005.42,0.0,0.0,10
4,2025-04-22 13:30:50+00:00,18004.11,18004.11,17998.31,18000.87,0.0,0.0,10


,date,open,high,low,close,volume,average,barCount
70165,2025-06-03 19:59:10+00:00,21675.37,21676.41,21674.05,21674.90,0.0,0.0,10
70166,2025-06-03 19:59:20+00:00,21675.04,21676.97,21675.04,21675.15,0.0,0.0,10
70167,2025-06-03 19:59:30+00:00,21674.99,21674.99,21668.30,21668.30,0.0,0.0,10
70168,2025-06-03 19:59:40+00:00,21669.02,21670.86,21667.66,21667.66,0.0,0.0,10
70169,2025-06-03 19:59:50+00:00,21666.57,21666.57,21661.19,21663.09,0.0,0.0,10


,date,open,high,low,close
0,2025-04-22 13:30:10+00:00,18033.90,18033.90,18019.76,18019.76
1,2025-04-22 13:30:20+00:00,18018.86,18018.86,18012.71,18012.74
2,2025-04-22 13:30:30+00:00,18013.64,18013.64,18002.60,18003.74
3,2025-04-22 13:30:40+00:00,18004.84,18007.12,18004.42,18005.42
4,2025-04-22 13:30:50+00:00,18004.11,18004.11,17998.31,18000.87


#### Sauvegarde du fichier

Construction du nom du fichier et sauvegarde en `.csv` dans le dossier `database`

Structure du nom du fichier : `Symbol_Interval_StartDate_EndDate_PriceSource.csv`


In [ ]:
# Récupérer la devise et l'unité de temps
symbol = contract.symbol

# Construire le nom du fichier
start_date = new_df['date'].iloc[0].strftime('%Y%m%d')

# Structure
save_path = f"../marketData/{symbol}_{historical_data_interval.replace(' ', '')}_{start_date}_to_{end_date}_{price_source}.parquet"

# Sauvegarder le DataFrame en fichier parquet
new_df.to_csv(save_path, index=True)
print(f"Fichier sauvegardé sous le nom : {save_path}")

#### Chargement d'un fichier

fichier csv

In [ ]:
import pandas as pd
import os
save_path = "../marketData/NDX_30secs_20220214_to_20250502_TRADES.csv"

retrieved_df = pd.read_csv(save_path)
display(retrieved_df.head())

In [ ]:
retrieved_df = resample_ohlc(retrieved_df, '1min')
retrieved_df

In [1]:
save_path = "../marketData/NDX_20secs_20220214_to_20250502_TRADES.csv"
retrieved_df.to_csv(save_path, index=True)
import pandas as pd
import os

retrieved_df = pd.read_csv(save_path)
display(retrieved_df.head())

NameError: name 'retrieved_df' is not defined

In [ ]:
retrieved_df.to_parquet(save_path, index=True, compression=None)

## Additional features

#### Data pre processing

In [8]:
import pandas as pd
from Helpers import resample_ohlc

interval_target = '1min'  # Interval cible pour le resampling

original_df = '../marketData/NDX_10secs_20220214_to_20250604_TRADES.csv'
save_path = f'../marketData/NDX_{interval_target}_20220214_to_20250604_TRADES.csv'

original_df = pd.read_csv(original_df)
display(original_df.head())
resampled_df = resample_ohlc(original_df, interval_target)
resampled_df.to_csv(save_path, index=True)
print(f"Fichier sauvegardé sous le nom : {save_path}")
display(resampled_df.head())

,date,open,high,low,close
0,2022-02-14 14:30:10+00:00,14233.40,14239.47,14233.40,14239.47
1,2022-02-14 14:30:20+00:00,14239.47,14242.25,14231.47,14235.56
2,2022-02-14 14:30:30+00:00,14235.56,14242.16,14235.56,14239.24
3,2022-02-14 14:30:40+00:00,14239.24,14239.45,14233.44,14236.41
4,2022-02-14 14:30:50+00:00,14236.41,14257.50,14236.41,14257.50


Fichier sauvegardé sous le nom : ../marketData/NDX_1min_20220214_to_20250604_TRADES.csv


,date,open,high,low,close
0,2022-02-14 14:30:00+00:00,14233.40,14257.50,14231.47,14257.50
1,2022-02-14 14:31:00+00:00,14257.50,14267.11,14238.47,14238.47
2,2022-02-14 14:32:00+00:00,14238.47,14238.47,14209.07,14209.81
3,2022-02-14 14:33:00+00:00,14209.81,14214.50,14196.92,14198.70
4,2022-02-14 14:34:00+00:00,14198.70,14238.12,14198.41,14238.12


#### DataFrame update 
Update the dataframe by merging the new datas with old ones

In [8]:
import pandas as pd
from Helpers import merge_ohlc_dataframes

# Load your existing data
existing_file_path = "../marketData/NDX_10secs_20220214_to_20250502_TRADES.csv"
existing_df = pd.read_csv(existing_file_path)

trading_hours = {'start' : '13:30:00', 'end' : '20:00:00'}  # Define your trading hours

# Merge the dataframes
merged_df = merge_ohlc_dataframes(existing_df, new_df, frequency='10s', trading_hours=trading_hours)

# Save the merged dataframe
save_path = f"../marketData/{contract.symbol}_10secs_20220214_to_{end_date}_{price_source}.csv"
display(merged_df.head())
display(merged_df.tail())
merged_df.to_csv(save_path, index=False)
print(f"Merged data saved to: {save_path}")

Checking for gaps only during trading hours: 13:30:00 to 20:00:00 UTC
First few missing timestamps: [Timestamp('2022-02-15 13:31:00'), Timestamp('2022-02-15 13:31:10'), Timestamp('2022-02-15 13:31:20'), Timestamp('2022-02-15 13:31:30'), Timestamp('2022-02-15 13:31:40')]


,date,open,high,low,close
0,2022-02-14 14:30:10+00:00,14233.40,14239.47,14233.40,14239.47
1,2022-02-14 14:30:20+00:00,14239.47,14242.25,14231.47,14235.56
2,2022-02-14 14:30:30+00:00,14235.56,14242.16,14235.56,14239.24
3,2022-02-14 14:30:40+00:00,14239.24,14239.45,14233.44,14236.41
4,2022-02-14 14:30:50+00:00,14236.41,14257.50,14236.41,14257.50


,date,open,high,low,close
1930208,2025-06-03 19:59:10+00:00,21675.37,21676.41,21674.05,21674.90
1930209,2025-06-03 19:59:20+00:00,21675.04,21676.97,21675.04,21675.15
1930210,2025-06-03 19:59:30+00:00,21674.99,21674.99,21668.30,21668.30
1930211,2025-06-03 19:59:40+00:00,21669.02,21670.86,21667.66,21667.66
1930212,2025-06-03 19:59:50+00:00,21666.57,21666.57,21661.19,21663.09


Merged data saved to: ../marketData/NDX_10secs_20220214_to_20250604-07:44:04_TRADES.csv


In [ ]:
import pandas as pd

symbol = 'NDX'
interval = '10secs'
start_date = '20220214'
price_source= 'TRADES'

# 1. Load the existing dataframe
existing_file_path = "../marketData/NDX_10secs_20220214_to_20250411_TRADES.parquet"
existing_df = pd.read_parquet(existing_file_path)

# 3. Check for the last timestamp in existing data
last_timestamp = existing_df['date'].max()
print(f"Last timestamp in existing data: {last_timestamp}")

updated_df_path = "../marketData/NDX_10secs_20250403_to_20250502_TRADES.parquet"
updated_df = pd.read_parquet(updated_df_path)

# 5. Concatenate the dataframes
merged_df = pd.concat([existing_df, updated_df])

# 6. Sort the merged dataframe by date
merged_df = merged_df.sort_values('date')

# 7. Reset the index to create a clean sequential index
merged_df = merged_df.reset_index(drop=True)

# Display summary
print(f"Original data points: {len(existing_df)}")
print(f"New data points: {len(updated_df)}")
print(f"Total data points after merge: {len(merged_df)}")
merged_df

In [ ]:
save_path = f"../marketData/{symbol}_{interval}_{start_date}_to_{end_date}_{price_source}.parquet"
merged_df.to_parquet(save_path, index=True, compression=None)
print(f"Fichier sauvegardé sous le nom : {save_path}")